# Make Event Selection Plots for Technote

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot
from matplotlib import gridspec

import sys
sys.path.append('../../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
import pyanalib.stat_helpers as sh
from makedf.util import *
from analysis_village.plot_style.plot_helper import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
sample_str = "v10_14_02_02+"

## Open Data Frames

In [ ]:
input_path = "/data/sungbino/sbnd/gen2/cohpi/"
mc_file_path = input_path + "aurora_SBND2026A_gen2_BNBLight_prodgenie_corsika_proton_rockbox0p1_sbnd_CV_v10_14_02_03_flatcaf_sbnd_cohpi_slcdf_500files.df"
bnb_light_data_file_path = input_path + "data_SBND2026A_SBND2026A_gen2_run1_BNBLight_Data_FixedDev_Respin_v10_14_02_04_flatcaf_sbnd_cohpi_slcdf.df"
offbeam_light_data_file_path = input_path + "data_SBND2026A_gen2_InTime-Run1_v10_14_02_02_flatcaf_sbnd_cohpi_slcdf.df"

In [ ]:
print("mc n split", get_n_split(mc_file_path))
print("bnb +light data n split", get_n_split(bnb_light_data_file_path))
print("offbeam +light data n split", get_n_split(offbeam_light_data_file_path))
print("Keys in mc file:")
print_keys(mc_file_path)

In [ ]:
keys2load = ['hdr', "mcnu", 'evt', "pot"]
mc_bnb_cosmic_dfs = load_dfs(mc_file_path, keys2load, n_max_concat=3)
data_bnb_light_dfs = load_dfs(bnb_light_data_file_path, keys2load, n_max_concat=3)
data_offbeam_light_dfs = load_dfs(offbeam_light_data_file_path, keys2load, n_max_concat=3)

## Exposure Accounting

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect the offbeam data fudge factor and scale for offbeam data
n_record_spill_data = get_n_evt(data_bnb_light_dfs['hdr'])
n_gates_data = len(data_bnb_light_dfs["pot"])

n_record_spill_offbeam_data = get_n_evt(data_offbeam_light_dfs['hdr'])
n_gates_offbeam_data = data_offbeam_light_dfs["hdr"][data_offbeam_light_dfs["hdr"]['first_in_subrun'] == 1]['noffbeambnb'].sum()

p_trig_data = n_record_spill_data / n_gates_data
p_trig_offbeam_data = n_record_spill_offbeam_data / n_gates_offbeam_data
print("p_trig_data: %f" %p_trig_data)
print("p_trig_offbeam_data: %f" %p_trig_offbeam_data)

f_factor = (p_trig_data - p_trig_offbeam_data) / (1 - p_trig_offbeam_data)
print("f_factor: %f" %f_factor)

intime_gate_scale = (1. - f_factor) * (n_gates_data + 0.) / (n_gates_offbeam_data + 0.)
print("intime_gate_scale: %f" %intime_gate_scale)


In [ ]:
## Collect pot scale for MC
mc_tot_pot = mc_bnb_cosmic_dfs["hdr"]['pot'].sum()

data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))

print("data_tot_pot: %e" %(data_tot_pot))
print("data_tot_TOR860: %e" %(data_tot_TOR860))
print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))

In [ ]:
## Comparison between observed and expected total number of recorded spills
n_evt_mc = get_n_evt(mc_bnb_cosmic_dfs["hdr"])

print("n_evt_data_onbeam: %d" %n_record_spill_data)
print("n_evt_exp.: %f" %(n_evt_mc * mc_pot_scale + n_record_spill_offbeam_data * intime_gate_scale))
print("- n_evt_mc: %f" %(n_evt_mc * mc_pot_scale))
print("- n_evt_data_offbeam: %f" %(n_record_spill_offbeam_data * intime_gate_scale))

## Matching between MC reco and Truth df

In [ ]:
# match for mc bnb cosmic
mc_bnb_cosmic_dfs["evt"].columns = pd.MultiIndex.from_tuples(
    [(*col, '') if isinstance(col, tuple) else (col, '') 
     for col in mc_bnb_cosmic_dfs["evt"].columns]
)
mc_bnb_cosmic_dfs["evt"] = ph.multicol_merge(mc_bnb_cosmic_dfs["evt"].reset_index(), mc_bnb_cosmic_dfs["mcnu"].reset_index(),
                            left_on=[('__ntuple', ''), ('entry', ''), ('tmatch_idx','')],
                            right_on=[('__ntuple', ''), ('entry', ''), ('rec.mc.nu..index', '')], 
                            how="left") ## -- save all sllices
mc_bnb_cosmic_dfs["evt"] = mc_bnb_cosmic_dfs["evt"].set_index(["__ntuple", "entry", "rec.slc..index"], verify_integrity=True)
mc_bnb_cosmic_dfs["evt"].loc[mc_bnb_cosmic_dfs["evt"][('nuint_categ', '')].isna(), [('nuint_categ', '')]] = -2

data_offbeam_light_dfs["evt"].columns = pd.MultiIndex.from_tuples(
    [(*col, '') if isinstance(col, tuple) else (col, '') 
     for col in data_offbeam_light_dfs["evt"].columns]
)
data_offbeam_light_dfs["evt"][('nuint_categ', '')] = -3

data_bnb_light_dfs["evt"].columns = pd.MultiIndex.from_tuples(
    [(*col, '') if isinstance(col, tuple) else (col, '') 
     for col in data_bnb_light_dfs["evt"].columns]
)

## Plotter

In [ ]:
mode_list = [1, 2, 0, 3, 4, 5, 6, -1, -2, -3]
mode_labels = ["Signal", "Non-Sig. CCCOH", "NC", "QE", "2p2h", "RES", "DIS", "Non-FV", "Others", "Intime Cosmics"]
colors = ['#d62728',  # Red            
          '#1f77b4',  # Blue
          '#ff7f0e',  # Orange
          '#2ca02c',  # Green
          '#17becf',  # Teal
          '#9467bd',  # Purple
          '#8c564b',  # Brown
          '#e377c2',  # Pink
          '#7f7f7f',  # Gray
          '#bcbd22',  # Yellow-green
]  # Gold

def draw_reco_stacked_hist(var_mc, var_offbeam_data, is_logx, is_logy,
                           title_x, title_y, x_min, x_max, nbins, outname,
                           data_overlay=False, var_data=[], draw_density=False):
    
    ## Define the output figure to have two pads
    fig = plt.figure(figsize=(8, 8), dpi=100)
    gs = gridspec.GridSpec(2, 1, height_ratios=[5, 1], hspace=0.10)
    ax_main = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_main)
    
    if is_logx:
        ax_main.set_xscale('log')
        ax_ratio.set_xscale('log')        
    if is_logy:
        ax_main.set_yscale('log')


    ax_main.set_xlabel("")  # Only bottom has x-label
    ax_main.set_ylabel(title_y)
    ax_ratio.set_ylabel("Data/MC", fontsize=12)
    ax_ratio.set_xlabel(title_x, fontsize = 20)

    ax_ratio.axhline(1.0, color='red', linestyle='--', linewidth=1)
    ax_ratio.set_ylabel("Data/MC", fontsize=12)
    ax_ratio.set_xlabel(title_x, fontsize=12)
    ax_ratio.set_ylim(0.5, 1.5)
    ax_ratio.tick_params(width=2, length=6)
    for spine in ax_ratio.spines.values():
        spine.set_linewidth(2)

    plt.setp(ax_main.get_xticklabels(), visible=False)

    ## Defind binning
    if is_logx:
        bins = np.logspace(np.log10(x_min), np.log10(x_max), nbins + 1)
        bin_centers = np.sqrt(bins[:-1] * bins[1:])
    else:
        bins = np.linspace(x_min, x_max, nbins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])

    ## Define data for MC
    all_mc_data = var_mc + var_offbeam_data
    all_weights = (
        [np.ones_like(data) * mc_pot_scale for data in var_mc] +
        [np.ones_like(data) * intime_gate_scale for data in var_offbeam_data]
    )
    each_mc_hist_data = [np.histogram(data, bins=bins, weights=w)[0] for data, w in zip(all_mc_data, all_weights)]
    total_mc = np.sum(each_mc_hist_data, axis=0)

    ## Plot stacked MC
    hist_data, bins, _ = ax_main.hist(all_mc_data,
                                      bins=bins,
                                      weights=all_weights,
                                      stacked=True,
                                      color=colors,
                                      label=mode_labels,
                                      edgecolor='none',
                                      linewidth=0,
                                      density=draw_density,
                                      histtype='stepfilled')

    max_y = np.max(total_mc)

    ## Plot MC stat error box
    each_mc_hist_data = []
    each_mc_hist_err2 = []  # sum of squared weights for error

    for data, w in zip(all_mc_data, all_weights):
        hist_vals, _ = np.histogram(data, bins=bins, weights=w)
        hist_err2, _ = np.histogram(data, bins=bins, weights=np.square(w))
        each_mc_hist_data.append(hist_vals)
        each_mc_hist_err2.append(hist_err2)

    total_mc = np.sum(each_mc_hist_data, axis=0)
    total_mc_err2 = np.sum(each_mc_hist_err2, axis=0)
    mc_stat_err = np.sqrt(total_mc_err2)
    #mc_stat_err = np.sqrt(total_mc)

    ax_main.bar(
       bin_centers,
        2 * mc_stat_err,
        width=np.diff(bins),
        bottom=total_mc - mc_stat_err,
        facecolor='none',             # transparent fill
        edgecolor='black',            # outline color of the hatching
        hatch='xxxx',                 # hatch pattern similar to ROOT's 3004
        linewidth=0.0,
        label='MC Stat. Unc.'
    )

    ax_main.tick_params(width=2, length=10)
    for spine in ax_main.spines.values():
        spine.set_linewidth(2)

    ## Draw Ratio error bar
    mc_stat_err_ratio = mc_stat_err / total_mc
    mc_content_ratio = total_mc / total_mc
    mc_stat_err_ratio = np.nan_to_num(mc_stat_err_ratio, nan=0.)
    mc_content_ratio = np.nan_to_num(mc_content_ratio, nan=-999.)
    ax_ratio.bar(
        bin_centers,
        2*mc_stat_err_ratio,
        width=np.diff(bins),
        bottom=mc_content_ratio - mc_stat_err_ratio,
        facecolor='none',             # transparent fill
        edgecolor='black',            # outline color of the hatching
        hatch='xxxx',                 # hatch pattern similar to ROOT's 3004
        linewidth=0.0,
        label='MC Stat. Unc.'
    )

    ## Draw data too
    if data_overlay:
        ax_main.set_ylabel("Events (POT = %.2e)" % target_pot)
        if draw_density:
            ax_main.set_ylabel("A.U.")

        ## Define data histogram
        counts, _ = np.histogram(var_data, bins=bins)

        bin_widths = np.diff(bins)
        total_data = np.sum(counts)
        norm_counts = counts
        data_eylow, data_eyhigh = sh.return_data_stat_err(counts)

        if draw_density:
            norm_counts = counts / (total_data * bin_widths)
            data_eylow = data_eylow / (total_data * bin_widths) if total_data > 0 else np.zeros_like(counts)
            data_eyhigh = data_eyhigh / (total_data * bin_widths) if total_data > 0 else np.zeros_like(counts)

        errors = data_eylow + data_eyhigh
        
        ## Plot data points on main histogram
        #ax_main.errorbar(bin_centers, norm_counts, yerr=errors,
        #                 fmt='o', color='black', label='Data',
        #                 markersize=5, capsize=3, linewidth=1.5)
        
        ax_main.errorbar(bin_centers, norm_counts,
                 yerr=np.vstack((data_eylow, data_eyhigh)),
                 fmt='o', color='black', label='Data',
                 markersize=5, capsize=3, linewidth=1.5)
        
        max_y_data = np.max(norm_counts + data_eyhigh)
        #print("max_y: %f" %(max_y))
        #print("max_y_data: %f" %(max_y_data))
        max_y = max(max_y, max_y_data)
        #print("max_y: %f" %(max_y))

        ## Make data/mc ratio plot
        data_ratio = norm_counts / total_mc
        data_ratio_eylow = data_eylow / total_mc
        data_ratio_eyhigh = data_eyhigh / total_mc
        data_ratio = np.nan_to_num(data_ratio, nan=-999.)
        data_ratio_eylow = np.nan_to_num(data_ratio_eylow, nan=0.)
        data_ratio_eyhigh = np.nan_to_num(data_ratio_eyhigh, nan=0.)
        
        #data_ratio_errors = data_ratio_eylow + data_ratio_eyhigh
        #ax_ratio.errorbar(bin_centers, data_ratio, yerr=data_ratio_errors,
        #                 fmt='o', color='black', label='Data',
        #                 markersize=5, capsize=3, linewidth=1.5)

        ax_ratio.errorbar(bin_centers, data_ratio,
                  yerr=np.vstack((data_ratio_eylow, data_ratio_eyhigh)),
                  fmt='o', color='black', label='Data',
                  markersize=5, capsize=3, linewidth=1.5)

    ## Set ax_main axes variables
    ax_main.set_xlim(x_min, x_max)
    ax_main.set_ylim(0.0, max_y * 1.5)
    if is_logy:
        ax_main.set_ylim(0.01, max_y * 600)
    
    # Legend with fractions
    accum_sum = [np.sum(data) for data in hist_data]
    accum_sum = [0.] + accum_sum
    total_sum = accum_sum[-1]
    individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
    fractions = [(count / total_sum) * 100 for count in individual_sums]
    legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
    if data_overlay:
        if draw_density:
            legend_labels.append("Data")
        else:
            legend_labels.append(f"Total MC Stat. Unc. ({total_sum:.0f})")
            legend_labels.append(f"Data ({total_data:.0f})")
            #legend_labels.append(f"Data ({total_data:.0f})")
    ax_main.legend(legend_labels, loc='upper left', fontsize=8, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

    legend_labels_ratio = ["y=1", "MC (Stat. Only)", "Data/MC"]
    ax_ratio.legend(legend_labels_ratio, loc='upper left', fontsize=6, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))
    ax_ratio.minorticks_on()

    ax_main.text(0.00, 1.02, "SBND " + sample_str + ", Internal",
                 transform=ax_main.transAxes, fontsize=14, fontweight='bold')

    fig.savefig("./" + outname + ".pdf", format='pdf', bbox_inches='tight')
    plt.show()
    plt.close()

def draw_mc_data_shape_comp_per_slc(mc_bnb_cosmic_df, data_offbeam_df, data_df, column, x_title, y_title, x_min, x_max, n_bins, out_name, is_logx = False, is_logy = False):
    nuint_categ_col = ('nuint_categ', '')

    mc_bnb_cosmic_df_per_slc = mc_bnb_cosmic_df.groupby([('__ntuple'), ('entry'), ('rec.slc..index')])[[column, nuint_categ_col]].first()
    data_offbeam_df_per_slc = data_offbeam_df.groupby([('__ntuple'), ('entry'), ('rec.slc..index')])[[column, nuint_categ_col]].first()

    data_df_per_slc = data_df.groupby([('__ntuple'), ('entry'), ('rec.slc..index')])[[column]].first()

    mode_list_mc = [m for m in mode_list if ((m != -3) and (m != -4))]
    var_mc_bnb_cosmic = [mc_bnb_cosmic_df_per_slc[mc_bnb_cosmic_df_per_slc[nuint_categ_col] == mode][column]for mode in mode_list_mc]
    var_data_offbeam = [data_offbeam_df_per_slc[data_offbeam_df_per_slc.nuint_categ == -3][column]]
    var_data = data_df_per_slc[column]
    draw_reco_stacked_hist(var_mc_bnb_cosmic, var_data_offbeam, is_logx, is_logy, x_title, y_title, x_min, x_max, n_bins, out_name, True, var_data)

In [ ]:
def draw_reco_stacked_hist_blind(
    var_mc, var_offbeam_data, is_logx, is_logy,
    title_x, title_y, x_min, x_max, nbins, outname,
    data_overlay=False, var_data=[], draw_density=False,
    blind=True, blind_req=0.1, label_top_right=None
):
    fig = plt.figure(figsize=(8, 8), dpi=100)
    gs = gridspec.GridSpec(2, 1, height_ratios=[5, 1], hspace=0.10)
    ax_main = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_main)

    if is_logx:
        ax_main.set_xscale('log')
        ax_ratio.set_xscale('log')
    if is_logy:
        ax_main.set_yscale('log')

    ax_main.set_xlabel("")
    ax_main.set_ylabel(title_y)
    ax_ratio.set_ylabel("Data/MC", fontsize=12)
    ax_ratio.set_xlabel(title_x, fontsize=12)
    ax_ratio.axhline(1.0, color='red', linestyle='--', linewidth=1)
    ax_ratio.set_ylim(0.4, 1.6)
    
    ax_ratio.tick_params(width=2, length=6)
    for spine in ax_ratio.spines.values():
        spine.set_linewidth(2)

    plt.setp(ax_main.get_xticklabels(), visible=False)

    bins = np.logspace(np.log10(x_min), np.log10(x_max), nbins + 1) if is_logx else np.linspace(x_min, x_max, nbins + 1)
    bin_centers = np.sqrt(bins[:-1] * bins[1:]) if is_logx else 0.5 * (bins[:-1] + bins[1:])

    all_mc_data = var_mc + var_offbeam_data
    all_weights = (
        [np.ones_like(data) * mc_pot_scale for data in var_mc] +
        [np.ones_like(data) * intime_gate_scale for data in var_offbeam_data]
    )

    each_mc_hist_data = [
        np.histogram(data, bins=bins, weights=w)[0]
        for data, w in zip(all_mc_data, all_weights)
    ]
    total_mc = np.sum(each_mc_hist_data, axis=0)

    hist_data, bins, _ = ax_main.hist(
        all_mc_data, bins=bins, weights=all_weights,
        stacked=True, color=colors, label=mode_labels,
        edgecolor='none', linewidth=0, density=draw_density, histtype='stepfilled'
    )

    max_y = np.max(total_mc)
    each_mc_hist_data = []
    each_mc_hist_err2 = []  # sum of squared weights for error

    for data, w in zip(all_mc_data, all_weights):
        hist_vals, _ = np.histogram(data, bins=bins, weights=w)
        hist_err2, _ = np.histogram(data, bins=bins, weights=np.square(w))
        each_mc_hist_data.append(hist_vals)
        each_mc_hist_err2.append(hist_err2)

    total_mc = np.sum(each_mc_hist_data, axis=0)
    total_mc_err2 = np.sum(each_mc_hist_err2, axis=0)
    mc_stat_err = np.sqrt(total_mc_err2)

    ax_main.bar(
        bin_centers, 2 * mc_stat_err,
        width=np.diff(bins), bottom=total_mc - mc_stat_err,
        facecolor='none', edgecolor='black', hatch='xxxx',
        linewidth=0.0, label='MC Stat. Unc.'
    )

    ax_main.tick_params(width=2, length=10)
    for spine in ax_main.spines.values():
        spine.set_linewidth(2)

    mc_stat_err_ratio = np.nan_to_num(mc_stat_err / total_mc, nan=0.)
    mc_content_ratio = np.nan_to_num(total_mc / total_mc, nan=-999.)
    
    ax_ratio.bar(
        bin_centers, 2 * mc_stat_err_ratio,
        width=np.diff(bins), bottom=mc_content_ratio - mc_stat_err_ratio,
        facecolor='none', edgecolor='black', hatch='xxxx',
        linewidth=0.0, label='MC Stat. Unc.'
    )

    if data_overlay:
        ax_main.set_ylabel("A.U." if draw_density else f"Events (POT = {target_pot:.2e})")

        counts, _ = np.histogram(var_data, bins=bins)
        bin_widths = np.diff(bins)
        total_data = np.sum(counts)

        norm_counts = counts.copy() # Use copy to safely modify below
        data_eylow, data_eyhigh = sh.return_data_stat_err(counts)
        
        if draw_density and total_data > 0:
            norm_counts = counts / (total_data * bin_widths)
            data_eylow = data_eylow / (total_data * bin_widths)
            data_eyhigh = data_eyhigh / (total_data * bin_widths)
        elif draw_density:
            norm_counts[:] = 0
            data_eylow[:] = 0
            data_eyhigh[:] = 0

        # Apply blinding to bins where Signal fraction > blind_req
        if blind:
            signal_mc = each_mc_hist_data[0]  # index 0 is assumed to be "Signal"
            signal_frac = np.nan_to_num(signal_mc / total_mc, nan=0.)
            blind_bins = signal_frac > blind_req
            norm_counts[blind_bins] = -1
            data_eylow[blind_bins] = 0
            data_eyhigh[blind_bins] = 0

        ax_main.errorbar(
            bin_centers, norm_counts,
            yerr=np.vstack((data_eylow, data_eyhigh)),
            fmt='o', color='black', label='Data',
            markersize=5, capsize=3, linewidth=1.5
        )

        max_y_data = np.max(norm_counts + data_eyhigh)
        max_y = max(max_y, max_y_data)

        data_ratio = np.nan_to_num(norm_counts / total_mc, nan=-1.)
        data_ratio_eylow = np.nan_to_num(data_eylow / total_mc, nan=0.)
        data_ratio_eyhigh = np.nan_to_num(data_eyhigh / total_mc, nan=0.)

        ax_ratio.errorbar(
            bin_centers, data_ratio,
            yerr=np.vstack((data_ratio_eylow, data_ratio_eyhigh)),
            fmt='o', color='black', label='Data',
            markersize=5, capsize=3, linewidth=1.5
        )

    ax_main.set_xlim(x_min, x_max)
    ax_main.set_ylim(0.01 if is_logy else 0.0, max_y * (600 if is_logy else 1.5))

    accum_sum = [np.sum(data) for data in hist_data]
    accum_sum = [0.] + accum_sum
    total_sum = accum_sum[-1]
    individual_sums = [accum_sum[i+1] - accum_sum[i] for i in range(len(accum_sum)-1)]
    fractions = [(count / total_sum) * 100 for count in individual_sums]
    legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]

    if data_overlay:
        if draw_density:
            legend_labels.append("Data")
        else:
            legend_labels.append(f"Total MC Stat. Unc. ({total_sum:.0f})")
            legend_labels.append(f"Data ({total_data:.0f})")

    ax_main.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))
    ax_ratio.legend(["y=1", "MC (Stat. Only)", "Data/MC"], loc='upper left', fontsize=7, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))
    ax_ratio.minorticks_on()

    ax_main.text(0.00, 1.02, "SBND " + sample_str + ", Internal",
                 transform=ax_main.transAxes, fontsize=14, fontweight='bold')
    if label_top_right is not None:
        ax_main.text(1.0, 1.02, label_top_right,
                     transform=ax_main.transAxes, fontsize=14, fontweight='bold', ha='right')

    fig.savefig(f"./{outname}.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    plt.close()

def draw_mc_data_shape_comp_per_slc_blind(
    mc_bnb_cosmic_df, data_offbeam_df, data_df, 
    column, x_title, y_title, x_min, x_max, n_bins, out_name, 
    is_logx=False, is_logy=False, blind=True, blind_req=0.1, label_top_right=None
):
    nuint_categ_col = ('nuint_categ', '')

    mc_bnb_cosmic_df_per_slc = mc_bnb_cosmic_df.groupby([('__ntuple'), ('entry'), ('rec.slc..index')])[[column, nuint_categ_col]].first()
    data_offbeam_df_per_slc = data_offbeam_df.groupby([('__ntuple'), ('entry'), ('rec.slc..index')])[[column, nuint_categ_col]].first()
    data_df_per_slc = data_df.groupby([('__ntuple'), ('entry'), ('rec.slc..index')])[[column]].first()

    mode_list_mc = [m for m in mode_list if ((m != -3) and (m != -4))]
    var_mc_bnb_cosmic = [mc_bnb_cosmic_df_per_slc[mc_bnb_cosmic_df_per_slc[nuint_categ_col] == mode][column] for mode in mode_list_mc]
    var_data_offbeam = [data_offbeam_df_per_slc[data_offbeam_df_per_slc.nuint_categ == -3][column]]
    var_data = data_df_per_slc[column]

    draw_reco_stacked_hist_blind(
        var_mc_bnb_cosmic, var_data_offbeam, is_logx, is_logy, 
        x_title, y_title, x_min, x_max, n_bins, out_name, 
        True, var_data, False, blind, blind_req, label_top_right
    )

## Evt sel plots

### Nocut: after vtx in FV and not clear cosmic cut

In [ ]:
vtx_x_col = ('vtx_x', '')
vtx_y_col = ('vtx_y', '')
vtx_z_col = ('vtx_z', '')
draw_mc_data_shape_comp_per_slc(mc_bnb_cosmic_dfs["evt"], data_offbeam_light_dfs["evt"], data_bnb_light_dfs["evt"], vtx_x_col, "Slice Vertex X [cm]",  "A.U.", -250, 250, 100,  "a_nocut_vtx_x", False, False)
draw_mc_data_shape_comp_per_slc(mc_bnb_cosmic_dfs["evt"], data_offbeam_light_dfs["evt"], data_bnb_light_dfs["evt"], vtx_y_col, "Slice Vertex Y [cm]",  "A.U.", -250, 250, 100,  "a_nocut_vtx_y", False, False)
draw_mc_data_shape_comp_per_slc(mc_bnb_cosmic_dfs["evt"], data_offbeam_light_dfs["evt"], data_bnb_light_dfs["evt"], vtx_z_col, "Slice Vertex Z [cm]",  "A.U.", -100, 600, 70,  "a_nocut_vtx_z", False, False)

### Check PE distribution and scale

In [ ]:
def abs_comp_total_pe_mc_stack(
    mc_series, data_bnb_light_series, data_offbeam_light_series,
    col_mc, col_data, title_x, x_min, x_max, nbins,
    title_y = "", is_logx = False, is_logy = False, label_top = "", label_SBND = "SBND Internal",
    mc_scale = 1.533, intime_scale = 1., out_name = None
):
    mc_var = mc_series
    data_bnb_light_var = data_bnb_light_series
    data_offbeam_light_var = data_offbeam_light_series

    # (kept for reference; override via intime_scale argument)
    data_offbeam_light_scale = 0.103

    # Binning
    bins = (np.logspace(np.log10(x_min), np.log10(x_max), nbins + 1)
            if is_logx else np.linspace(x_min, x_max, nbins + 1))
    bin_centers = (np.sqrt(bins[:-1] * bins[1:])
                   if is_logx else 0.5 * (bins[:-1] + bins[1:]))
    bin_widths = np.diff(bins)

    fig, ax = plt.subplots(figsize=(8, 5))

    # --- Data histogram (BNB light) ---
    ax.hist(
        data_bnb_light_var, bins=bins,
        histtype="step", color="black", linewidth=2,
        label="BNB + light data"
    )

    # --- Components: compute binned counts ---
    mc_hist, _ = np.histogram(mc_var, bins=bins)
    offbeam_hist, _ = np.histogram(data_offbeam_light_var, bins=bins)

    # Apply scales
    mc_hist = mc_hist * mc_scale
    offbeam_hist = offbeam_hist * intime_scale

    # --- Stack components as bars (to show contributions) ---
    # Bottoms for stacking
    bottom0 = np.zeros_like(mc_hist, dtype=float)
    h1 = mc_hist
    h3 = offbeam_hist

    # Use bars so stack works in both linear and log-x; they also respect log-y.
    # (We avoid specifying colors explicitly so you can theme globally if you like.)
    ax.bar(bins[:-1], h1, width=bin_widths, align="edge",
           alpha=0.45, edgecolor="none", label="MC (BNB+cosmic)")
    ax.bar(bins[:-1], h3, width=bin_widths, align="edge",
           bottom=h1, alpha=0.45, edgecolor="none", label="Off-beam light")

    # --- Total prediction overlay (dashed step) ---
    pred_hist = h1 + h3
    ax.step(bin_centers, pred_hist, where="mid", linewidth=2, linestyle="--",
            label="Total prediction", color='red')

    # Scales, labels, cosmetics
    if is_logx:
        ax.set_xscale("log")
    if is_logy:
        ax.set_yscale("log")

    ax.set_xlabel(title_x)
    ax.set_ylabel(title_y if title_y else "Events")
    ax.set_xlim(x_min, x_max)

    # Legend: put data first, then components, then total
    # (Matplotlib collects in order added; this already matches that.)
    ax.legend(frameon=False)

    if label_top:
        ax.text(1.0, 1.02, label_top, transform=ax.transAxes,
                fontsize=14, fontweight='bold', ha='right')
    if label_SBND:
        ax.text(0.02, 1.02, label_SBND, transform=ax.transAxes,
                fontsize=14, fontweight='bold', ha='left')

    plt.tight_layout()
    if out_name is not None:
        plt.savefig(out_name)
    plt.show()



def counts_above_thresholds(series, start=0.0, step=100.0, n_steps=30, comp=">"):
    """
    Compute counts of entries in `series` above each threshold.

    Parameters
    ----------
    series : pandas.Series or 1D array-like
        Values to threshold (e.g., df['totalpe_timecut']).
    start : float
        First threshold value.
    step : float
        Increment between thresholds.
    n_steps : int
        Number of thresholds to evaluate.
    comp : str
        Comparison: ">" or ">=".

    Returns
    -------
    thresholds : np.ndarray, shape (n_steps,)
    counts : np.ndarray, shape (n_steps,)
    """
    # Prepare data
    vals = np.asarray(series, dtype=float)
    vals = vals[~np.isnan(vals)]
    vals.sort()

    thresholds = start + step * np.arange(n_steps)

    if comp == ">":
        # count of values > t  ==  len(vals) - (# of values <= t)
        idx = np.searchsorted(vals, thresholds, side="right")
        counts = len(vals) - idx
    elif comp == ">=":
        # count of values >= t ==  len(vals) - (# of values < t)
        idx = np.searchsorted(vals, thresholds, side="left")
        counts = len(vals) - idx
    else:
        raise ValueError("comp must be '>' or '>='")

    return thresholds, counts

def plot_pe_threshold_comparison(
    data_series, data_intime_series, mc_series,
    label_str="MC Default",
    col_mc="totalpe_timecut",
    gate_scale=0.103, mc_pot_scale=0.312, mc_low_th_pot_scale=1.0,
    col="totalpe_timecut",
    start=0.0, step=1000.0, n_steps=50,
    ratio_ylim=(0.8, 1.2),  # y-range for ratio pad
    ratio_eps=1e-12,        # small guard to avoid 0-division
    out_name=None
):
    """
    Compare data vs (data_intime * gate_scale + MC * pot_scale + low_th_MC * pot_scale) above PE thresholds,
    and draw a bottom pad with the ratio data/model.
    """
    # Counts above thresholds
    data_pe_thr, data_pe_cnt = counts_above_thresholds(data_series, start, step, n_steps, comp=">")
    data_intime_pe_thr, data_intime_pe_cnt = counts_above_thresholds(data_intime_series, start, step, n_steps, comp=">")
    mc_pe_thr, mc_pe_cnt = counts_above_thresholds(mc_series, start, step, n_steps, comp=">")

    # Sanity check thresholds are aligned
    if not (np.allclose(data_pe_thr, data_intime_pe_thr) and np.allclose(data_pe_thr, mc_pe_thr)):
        raise ValueError("Threshold arrays differ. Check input parameters.")

    # Model
    model_cnt = (
        data_intime_pe_cnt * float(gate_scale)
        + mc_pe_cnt * float(mc_pot_scale)
    )

    # --- Figure with ratio pad ---
    fig, (ax, rax) = plt.subplots(
        2, 1, figsize=(7, 6.5), dpi=120,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
        sharex=True
    )

    # Top pad: counts
    ax.plot(data_pe_thr, data_pe_cnt, 'o-', label='BNB + Light Data', linewidth=1.5)
    ax.plot(mc_pe_thr, model_cnt, 's--', label='MC + Off-beam data (normalized)', linewidth=1.5)
    ax.plot(data_intime_pe_thr, data_intime_pe_cnt * float(gate_scale), 'o-', label="Off-beam (gates normalized)")
    ax.plot(mc_pe_thr, mc_pe_cnt * float(mc_pot_scale), 'o-', label="Default MC BNB + cosmic (POT normalized)")

    ax.set_ylabel('Spills (> threshold)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    ax.text(1.0, 1.02, label_str, transform=ax.transAxes, fontsize=14, fontweight='bold', ha='right')

    # y and x limits (same as your original, adjust if you like)
    # ymax = max(data_pe_cnt.max(), model_cnt.max())
    # ax.set_ylim(0, 1.5 * ymax)
    ax.set_ylim(0, 70000.)
    #ax.set_xlim(0., 20000.)
    ax.set_xlim(0., n_steps * step)  # ensure x-axis covers all thresholds

    # Bottom pad: ratio = data / model
    denom = np.maximum(model_cnt, ratio_eps)               # guard for zeros
    ratio = np.divide(data_pe_cnt, denom)                  # elementwise ratio
    # Optional: mask where model is ~0 so it doesn't mislead
    valid = model_cnt > ratio_eps

    rax.plot(data_pe_thr[valid], ratio[valid], 'o', linewidth=1.2, ms=2)
    rax.axhline(1.0, linestyle='--', linewidth=1.2, color='red')
    rax.set_ylim(*ratio_ylim)
    rax.set_ylabel('Data/Sum')
    rax.set_xlabel('PE threshold')
    rax.grid(True, alpha=0.3)

    ax.text(0.02, 1.02, "SBND Internal", transform=ax.transAxes,fontsize=14, fontweight='bold', ha='left')

    #plt.tight_layout()
    if out_name is not None:
        plt.savefig(out_name, bbox_inches='tight')
        #plt.savefig(out_name)
    plt.show()

    # Optionally return for further tweaking
    return fig, (ax, rax)

In [ ]:
mc_total_pe_per_spill = mc_bnb_cosmic_dfs["evt"].groupby(['__ntuple', 'entry'])['total_pe'].first()
data_bnb_light_total_pe_per_spill = data_bnb_light_dfs["evt"].groupby(['__ntuple', 'entry'])['total_pe'].first()
data_offbeam_light_total_pe_per_spill = data_offbeam_light_dfs["evt"].groupby(['__ntuple', 'entry'])['total_pe'].first()

In [ ]:
scales = {
    '1p01': 1.01, '1p02': 1.02, '1p03': 1.03, '1p04': 1.04, '1p05': 1.05, '1p06': 1.06, '1p07': 1.07, '1p08': 1.08, '1p09': 1.09, '1p10': 1.10,
    '1p11': 1.11, '1p12': 1.12, '1p13': 1.13
}

mc_total_pe_per_spill_scales = {suffix: mc_total_pe_per_spill * val for suffix, val in scales.items()}

In [ ]:
abs_comp_total_pe_mc_stack(
    mc_total_pe_per_spill, data_bnb_light_total_pe_per_spill, data_offbeam_light_total_pe_per_spill,
    col_mc='total_pe', col_data='total_pe', title_x='Total PE per spill', x_min=0, x_max=10000, nbins=100,
    title_y='Spills', is_logx=False, is_logy=False, label_top="Total PE per spill", out_name="./pe_cut/total_pe_per_spill_comparison.pdf", mc_scale=mc_pot_scale, intime_scale=intime_gate_scale)

#for suffix, mc_scaled in mc_total_pe_per_spill_scales.items():
#    abs_comp_total_pe_mc_stack(
#        mc_scaled, data_bnb_light_total_pe_per_spill, data_offbeam_light_total_pe_per_spill,
#        col_mc='total_pe', col_data='total_pe', title_x='Total PE per spill', x_min=0, x_max=40000, nbins=100,
#        title_y='Spills', is_logx=False, is_logy=False, label_top=f"Total PE per spill (MC scaled by {scales[suffix]:.2f})", out_name=f"./pe_cut/total_pe_per_spill_comparison_{suffix}", mc_scale=mc_pot_scale * scales[suffix], intime_scale=intime_gate_scale)

In [ ]:
plot_pe_threshold_comparison(data_bnb_light_total_pe_per_spill, data_offbeam_light_total_pe_per_spill, mc_total_pe_per_spill, ratio_ylim=(0.85, 1.05), start=0.0, step=10.0, n_steps=1000, gate_scale = intime_gate_scale, mc_pot_scale=mc_pot_scale, out_name="./pe_cut/totalpe_threshold_comparison_mc_1p0.pdf")

#for suffix, mc_scaled in mc_total_pe_per_spill_scales.items():
#    plot_pe_threshold_comparison(data_bnb_light_total_pe_per_spill, data_offbeam_light_total_pe_per_spill, mc_scaled, label_str=f"MC scaled by {scales[suffix]:.2f}", ratio_ylim=(0.85, 1.05), start=0.0, step=10.0, n_steps=6000, gate_scale = intime_gate_scale, mc_pot_scale=mc_pot_scale * scales[suffix], out_name=f"./pe_cut/totalpe_threshold_comparison_mc_{suffix}.pdf")

In [ ]:
total_pe_col = ('total_pe', '')
draw_mc_data_shape_comp_per_slc(mc_bnb_cosmic_dfs["evt"], data_offbeam_light_dfs["evt"], data_bnb_light_dfs["evt"], total_pe_col, "Total PE",  "A.U.", 0, 10000, 100,  "a_nocut_total_pe", False, False)
draw_mc_data_shape_comp_per_slc(mc_bnb_cosmic_dfs["evt"], data_offbeam_light_dfs["evt"], data_bnb_light_dfs["evt"], total_pe_col, "Total PE",  "A.U.", 0, 50000, 100,  "a_nocut_total_pe_log", False, True)


### PE > 1500 cut

In [ ]:
mc_evt_pe_2000 = mc_bnb_cosmic_dfs["evt"][mc_bnb_cosmic_dfs["evt"][('total_pe', '')] > 1500]
data_offbeam_light_evt_pe_2000 = data_offbeam_light_dfs["evt"][data_offbeam_light_dfs["evt"][('total_pe', '')] > 1500]
data_bnb_light_evt_pe_2000 = data_bnb_light_dfs["evt"][data_bnb_light_dfs["evt"][('total_pe', '')] > 1500]

In [ ]:
draw_mc_data_shape_comp_per_slc(mc_evt_pe_2000, data_offbeam_light_evt_pe_2000, data_bnb_light_evt_pe_2000, vtx_x_col, "Slice Vertex X [cm]",  "A.U.", -250, 250, 100,  "b_pe_cut_vtx_x", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_pe_2000, data_offbeam_light_evt_pe_2000, data_bnb_light_evt_pe_2000, vtx_y_col, "Slice Vertex Y [cm]",  "A.U.", -250, 250, 100,  "b_pe_cut_vtx_y", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_pe_2000, data_offbeam_light_evt_pe_2000, data_bnb_light_evt_pe_2000, vtx_z_col, "Slice Vertex Z [cm]",  "A.U.", -100, 600, 70,  "b_pe_cut_vtx_z", False, False)


In [ ]:
mc_evt_pe_2000.columns

### BCFM

In [ ]:
bcfm_score_col = ('bcfm_score', '')
draw_mc_data_shape_comp_per_slc(mc_evt_pe_2000, data_offbeam_light_evt_pe_2000, data_bnb_light_evt_pe_2000, bcfm_score_col, "BCFM Score",  "A.U.", 0, 0.5, 50,  "b_pe_cut_bcfm_score", False, True)

In [ ]:
mc_bcfm_score_0p05 = mc_evt_pe_2000[mc_evt_pe_2000[('bcfm_score', '')] > 0.05]
data_offbeam_light_evt_bcfm_0p05 = data_offbeam_light_evt_pe_2000[data_offbeam_light_evt_pe_2000[('bcfm_score', '')] > 0.05]
data_bnb_light_evt_bcfm_0p05 = data_bnb_light_evt_pe_2000[data_bnb_light_evt_pe_2000[('bcfm_score', '')] > 0.05]

In [ ]:
draw_mc_data_shape_comp_per_slc(mc_bcfm_score_0p05, data_offbeam_light_evt_bcfm_0p05, data_bnb_light_evt_bcfm_0p05, vtx_x_col, "Slice Vertex X [cm]",  "A.U.", -250, 250, 100, "c_bcfm_vtx_x", False, False)
draw_mc_data_shape_comp_per_slc(mc_bcfm_score_0p05, data_offbeam_light_evt_bcfm_0p05, data_bnb_light_evt_bcfm_0p05, vtx_y_col, "Slice Vertex Y [cm]",  "A.U.", -250, 250, 100, "c_bcfm_vtx_y", False, False)
draw_mc_data_shape_comp_per_slc(mc_bcfm_score_0p05, data_offbeam_light_evt_bcfm_0p05, data_bnb_light_evt_bcfm_0p05, vtx_z_col, "Slice Vertex Z [cm]",  "A.U.", -100, 600, 70, "c_bcfm_vtx_z", False, False)

### Nu-score

In [ ]:
nu_score_col = ('nu_score', '')
draw_mc_data_shape_comp_per_slc(mc_bcfm_score_0p05, data_offbeam_light_evt_bcfm_0p05, data_bnb_light_evt_bcfm_0p05, nu_score_col, "Nu Score",  "A.U.", 0, 1, 100,  "c_bcfm_nu_score", False, False)

### Two Prongs

In [ ]:
n_prongs_col = ('n_prong', '')
draw_mc_data_shape_comp_per_slc(mc_bcfm_score_0p05, data_offbeam_light_evt_bcfm_0p05, data_bnb_light_evt_bcfm_0p05, n_prongs_col, "Number of Prongs",  "A.U.", -0.5, 10.5, 11, "c_bcfm_n_prong", False, True)  

In [ ]:
mc_evt_n_prong_cut = mc_bcfm_score_0p05[mc_bcfm_score_0p05[('n_prong', '')] == 2]
data_offbeam_light_evt_n_prong_cut = data_offbeam_light_evt_bcfm_0p05[data_offbeam_light_evt_bcfm_0p05[('n_prong', '')] == 2]
data_bnb_light_evt_n_prong_cut = data_bnb_light_evt_bcfm_0p05[data_bnb_light_evt_bcfm_0p05[('n_prong', '')] == 2]

### N trks

In [ ]:
mc_evt_n_prong_cut

In [ ]:
n_trk_col = ('n_trk', '')
draw_mc_data_shape_comp_per_slc(mc_evt_n_prong_cut, data_offbeam_light_evt_n_prong_cut, data_bnb_light_evt_n_prong_cut, n_trk_col, "Number of Tracks",  "A.U.", -0.5, 3.5, 4, "d_n_prong_n_trk", False, True)

In [ ]:
mc_evt_n_trk_cut = mc_evt_n_prong_cut[mc_evt_n_prong_cut[('n_trk', '')] == 2]
data_offbeam_light_evt_n_trk_cut = data_offbeam_light_evt_n_prong_cut[data_offbeam_light_evt_n_prong_cut[('n_trk', '')] == 2]
data_bnb_light_evt_n_trk_cut = data_bnb_light_evt_n_prong_cut[data_bnb_light_evt_n_prong_cut[('n_trk', '')] == 2]

### Check chi2 pid

In [ ]:
long_trk_chi2_mu_col = ('long_trk_chi2_mu', '')
long_trk_chi2_pro_col = ('long_trk_chi2_pro', '')
long_trk_chi2_mu_new_col = ('long_trk_chi2_mu_new', '')
long_trk_chi2_pro_new_col = ('long_trk_chi2_pro_new', '')

short_trk_chi2_mu_col = ('short_trk_chi2_mu', '')
short_trk_chi2_pro_col = ('short_trk_chi2_pro', '')
short_trk_chi2_mu_new_col = ('short_trk_chi2_mu_new', '')
short_trk_chi2_pro_new_col = ('short_trk_chi2_pro_new', '')

draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, long_trk_chi2_mu_col, r"Long Track $\chi^{2}_\mu$",  "A.U.", 0, 80, 100, "e_n_trk_chi2_long_mu", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, long_trk_chi2_pro_col, r"Long Track $\chi^{2}_p$",  "A.U.", 0, 300, 100, "e_n_trk_chi2_long_pro", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, long_trk_chi2_mu_new_col, r"Long Track $\chi^{2}_\mu$ (New)",  "A.U.", 0, 80, 100, "e_n_trk_chi2_long_mu_new", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, long_trk_chi2_pro_new_col, r"Long Track $\chi^{2}_p$ (New)",  "A.U.", 0, 300, 100, "e_n_trk_chi2_long_pro_new", False, False)

draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, short_trk_chi2_mu_col, r"Short Track $\chi^{2}_\mu$",  "A.U.", 0, 80, 100, "e_n_trk_chi2_short_mu", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, short_trk_chi2_pro_col, r"Short Track $\chi^{2}_p$",  "A.U.", 0, 300, 100, "e_n_trk_chi2_short_pro", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, short_trk_chi2_mu_new_col, r"Short Track $\chi^{2}_\mu$ (New)",  "A.U.", 0, 80, 100, "e_n_trk_chi2_short_mu_new", False, False)
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, short_trk_chi2_pro_new_col, r"Short Track $\chi^{2}_p$ (New)",  "A.U.", 0, 300, 100, "e_n_trk_chi2_short_pro_new", False, True)

### Proton candidates multiplicity

In [ ]:
n_proton_col = ('n_proton', '')
draw_mc_data_shape_comp_per_slc(mc_evt_n_trk_cut, data_offbeam_light_evt_n_trk_cut, data_bnb_light_evt_n_trk_cut, n_proton_col, "Number of Protons",  "A.U.", -0.5, 3.5, 4, "e_n_trk_n_proton", False, True)

In [ ]:
mc_evt_n_proton_cut = mc_evt_n_trk_cut[mc_evt_n_trk_cut.n_proton == 0]
data_offbeam_light_evt_n_proton_cut = data_offbeam_light_evt_n_trk_cut[data_offbeam_light_evt_n_trk_cut.n_proton == 0]
data_bnb_light_evt_n_proton_cut = data_bnb_light_evt_n_trk_cut[data_bnb_light_evt_n_trk_cut.n_proton == 0]

### Shower multiplicities

In [ ]:
n_shower_col = ('n_shower', '')
draw_mc_data_shape_comp_per_slc(mc_evt_n_proton_cut, data_offbeam_light_evt_n_proton_cut, data_bnb_light_evt_n_proton_cut, n_shower_col, "Number of Showers",  "A.U.", -0.5, 5.5, 6, "f_n_proton_n_shower", False, True)

In [ ]:
mc_evt_n_shower_cut = mc_evt_n_proton_cut[mc_evt_n_proton_cut.n_shower == 0]
data_offbeam_light_evt_n_shower_cut = data_offbeam_light_evt_n_proton_cut[data_offbeam_light_evt_n_proton_cut.n_shower == 0]
data_bnb_light_evt_n_shower_cut = data_bnb_light_evt_n_proton_cut[data_bnb_light_evt_n_proton_cut.n_shower == 0]

### check truth distributions

In [ ]:
def plot_overlay_histograms(df, col1, col2, xmin = 0., xmax = 5., bins=30, density=True, out_name=None):
    """
    Compares the distributions of two dataframe columns using overlaid histograms.
    """
    plt.figure(figsize=(8, 5), dpi=100)
    plt.xlim(xmin, xmax)
    
    # Drop NaNs to prevent plotting errors
    data1 = df[col1].dropna()
    data2 = df[col2].dropna()
    
    # Plot both distributions
    plt.hist(data1, bins=bins, alpha=0.5, label=str(col1), density=density, edgecolor='black')
    plt.hist(data2, bins=bins, alpha=0.5, label=str(col2), density=density, edgecolor='black')
    
    # Formatting
    plt.xlabel('Value', fontsize=12)
    plt.ylabel('Density' if density else 'Counts', fontsize=12)
    plt.title(f'Distribution Comparison: {col1} vs {col2}', fontsize=14)
    
    # Add a grid and legend
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(loc='upper right', fontsize=10, frameon=False)
    
    # Save if an output name is provided
    if out_name is not None:
        plt.savefig(f"{out_name}.pdf", format='pdf', bbox_inches='tight')
        
    plt.show()
    plt.close()

import matplotlib.pyplot as plt

def plot_hist2d_colz(df, col_x, col_y, bins=50, cmap='viridis', out_name=None, val_min=0, val_max=1.5, x_title=None, y_title=None):
    """
    Creates a 2D histogram mimicking ROOT's 'COLZ' drawing option, with specified axis limits.
    """
    plt.figure(figsize=(8, 6), dpi=100)
    
    # Set the plot limits
    plt.xlim(val_min, val_max)
    plt.ylim(val_min, val_max)
    
    # Drop rows where either column has a NaN to prevent plotting errors
    plot_df = df[[col_x, col_y]].dropna()
    
    # Create the 2D histogram
    # range ensures the bins fit exactly within your specified min/max boundaries
    # cmin=1 leaves empty bins transparent (ROOT style)
    counts, xedges, yedges, im = plt.hist2d(
        plot_df[col_x], 
        plot_df[col_y], 
        bins=bins, 
        range=[[val_min, val_max], [val_min, val_max]],
        cmap=cmap, 
        cmin=1 
    )
    
    # Add the Z-axis color bar
    cbar = plt.colorbar(im)
    cbar.set_label('Events', fontsize=12)
    
    # Formatting
    plt.xlabel(x_title if x_title is not None else col_x, fontsize=12)
    plt.ylabel(y_title if y_title is not None else col_y, fontsize=12)
    
    # Clean up the title to use custom titles if provided
    title_x = x_title if x_title is not None else col_x
    title_y = y_title if y_title is not None else col_y
    plt.title("")

    # Add a light grid (zorder=0 pushes it behind the histogram)
    plt.grid(True, alpha=0.3, linestyle='--', zorder=0)
    
    plt.text(0.02, 1.02, "SBND v10_14_02_03 Internal", transform=plt.gca().transAxes, fontsize=14, fontweight='bold', ha='left')
    plt.text(1., 1.02, "Signal", transform=plt.gca().transAxes, fontsize=14, fontweight='bold', ha='right')


    # Save if an output name is provided
    if out_name is not None:
        plt.savefig(f"{out_name}.pdf", format='pdf', bbox_inches='tight')
        
    plt.show()
    plt.close()

In [ ]:
mc_evt_n_shower_cut_signal = mc_evt_n_shower_cut[mc_evt_n_shower_cut.nuint_categ == 1]

In [ ]:
mc_evt_n_shower_cut_signal

In [ ]:
plot_hist2d_colz(mc_evt_n_shower_cut_signal, 'true_p_mu', 'true_p_pi', bins=10, val_min=0, val_max=1.5, x_title='True Muon Momentum [GeV/c]', y_title='True Pion Momentum [GeV/c]', out_name="f_n_proton_n_shower_signal_true_p_mu_vs_true_p_pi")

### dir-Z

In [ ]:
long_dirz_col = ('long_dirz', '')
short_dirz_col = ('short_dirz', '')
draw_mc_data_shape_comp_per_slc(mc_evt_n_shower_cut, data_offbeam_light_evt_n_shower_cut, data_bnb_light_evt_n_shower_cut, long_dirz_col, r"$cos\theta_{\mu^\mp}$",  "A.U.", -1, 1, 50, "./no_dirz_cut/g_n_shower_long_dirz", False, True)
draw_mc_data_shape_comp_per_slc(mc_evt_n_shower_cut, data_offbeam_light_evt_n_shower_cut, data_bnb_light_evt_n_shower_cut, short_dirz_col, r"$cos\theta_{\pi^\pm}$",  "A.U.", -1, 1, 50, "./no_dirz_cut/g_n_shower_short_dirz", False, True)

In [ ]:
mc_evt_dirz_cut = mc_evt_n_shower_cut[(mc_evt_n_shower_cut.long_dirz > -1.) & (mc_evt_n_shower_cut.short_dirz > -1.)]
data_offbeam_light_evt_dirz_cut = data_offbeam_light_evt_n_shower_cut[(data_offbeam_light_evt_n_shower_cut.long_dirz > -1.) & (data_offbeam_light_evt_n_shower_cut.short_dirz > -1.)]
data_bnb_light_evt_dirz_cut = data_bnb_light_evt_n_shower_cut[(data_bnb_light_evt_n_shower_cut.long_dirz > -1.) & (data_bnb_light_evt_n_shower_cut.short_dirz > -1.)]

### Two track opening angle

In [ ]:
opening_angle_col = ('opening_angle', '')
draw_mc_data_shape_comp_per_slc_blind(mc_evt_dirz_cut, data_offbeam_light_evt_dirz_cut, data_bnb_light_evt_dirz_cut,
                                      opening_angle_col, r"$\rm{cos \theta_{\mu^\mp \pi^\pm}}$", "A.U.", -1, 1, 40, "./no_dirz_cut/h_dirz_opening_angle", 
                                      False, True, blind=True, blind_req=0.1)

In [ ]:
mc_evt_open_ang_cut = mc_evt_dirz_cut[mc_evt_dirz_cut.opening_angle > 0.5]
data_offbeam_light_evt_open_ang_cut = data_offbeam_light_evt_dirz_cut[data_offbeam_light_evt_dirz_cut.opening_angle > 0.5]
data_bnb_light_evt_open_ang_cut = data_bnb_light_evt_dirz_cut[data_bnb_light_evt_dirz_cut.opening_angle > 0.5]

### Further chi2 pid cuts

In [ ]:
long_trk_chi2_mu_new_col = ('long_trk_chi2_mu_new', '')
long_trk_chi2_pro_new_col = ('long_trk_chi2_pro_new', '')

short_trk_chi2_mu_new_col = ('short_trk_chi2_mu_new', '')
short_trk_chi2_pro_new_col = ('short_trk_chi2_pro_new', '')

draw_mc_data_shape_comp_per_slc_blind(mc_evt_open_ang_cut, data_offbeam_light_evt_open_ang_cut, data_bnb_light_evt_open_ang_cut, long_trk_chi2_mu_new_col, r"$\chi^{2}_\mu (\mu^\mp)$",  "A.U.", 0, 30, 50, "./no_dirz_cut/i_open_ang_chi2_long_mu_new", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_open_ang_cut, data_offbeam_light_evt_open_ang_cut, data_bnb_light_evt_open_ang_cut, long_trk_chi2_pro_new_col, r"$\chi^{2}_p (\mu^\mp)$",  "A.U.", 0, 300, 24, "./no_dirz_cut/i_open_ang_chi2_long_pro_new", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_open_ang_cut, data_offbeam_light_evt_open_ang_cut, data_bnb_light_evt_open_ang_cut, short_trk_chi2_mu_new_col, r"$\chi^{2}_\mu (\pi^\pm)$",  "A.U.", 0, 80, 50, "./no_dirz_cut/i_open_ang_chi2_short_mu_new", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_open_ang_cut, data_offbeam_light_evt_open_ang_cut, data_bnb_light_evt_open_ang_cut, short_trk_chi2_pro_new_col, r"$\chi^{2}_p (\pi^\pm)$",  "A.U.", 0, 300, 24, "./no_dirz_cut/i_open_ang_chi2_short_pro_new", False, True, blind=True, blind_req=0.1)

In [ ]:
mc_evt_chi2_proton_cut = mc_evt_open_ang_cut[(mc_evt_open_ang_cut.long_trk_chi2_pro_new > 100) & (mc_evt_open_ang_cut.short_trk_chi2_pro_new > 75)]
data_offbeam_light_evt_chi2_proton_cut = data_offbeam_light_evt_open_ang_cut[(data_offbeam_light_evt_open_ang_cut.long_trk_chi2_pro_new > 100) & (data_offbeam_light_evt_open_ang_cut.short_trk_chi2_pro_new > 75)]
data_bnb_light_evt_chi2_proton_cut = data_bnb_light_evt_open_ang_cut[(data_bnb_light_evt_open_ang_cut.long_trk_chi2_pro_new > 100) & (data_bnb_light_evt_open_ang_cut.short_trk_chi2_pro_new > 75)]

In [ ]:
draw_mc_data_shape_comp_per_slc_blind(mc_evt_chi2_proton_cut, data_offbeam_light_evt_chi2_proton_cut, data_bnb_light_evt_chi2_proton_cut, long_trk_chi2_mu_new_col, r"$\chi^{2}_\mu (\mu^\mp)$",  "A.U.", 0, 30, 50, "./no_dirz_cut/j_chi2_proton_chi2_long_mu_new", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_chi2_proton_cut, data_offbeam_light_evt_chi2_proton_cut, data_bnb_light_evt_chi2_proton_cut, short_trk_chi2_mu_new_col, r"$\chi^{2}_\mu (\pi^\pm)$",  "A.U.", 0, 80, 50, "./no_dirz_cut/j_chi2_proton_chi2_short_mu_new", False, True, blind=True, blind_req=0.1)


In [ ]:
mc_evt_chi2_proton_cut

### charge spheres

In [ ]:
mc_evt_chi2_proton_cut.sum_integ_1cm = mc_evt_chi2_proton_cut.sum_integ_1cm.fillna(0.)
mc_evt_chi2_proton_cut.sum_integ_2cm = mc_evt_chi2_proton_cut.sum_integ_2cm.fillna(0.)
mc_evt_chi2_proton_cut.sum_integ_3cm = mc_evt_chi2_proton_cut.sum_integ_3cm.fillna(0.)
mc_evt_chi2_proton_cut.sum_integ_4cm = mc_evt_chi2_proton_cut.sum_integ_4cm.fillna(0.)

data_offbeam_light_evt_chi2_proton_cut.sum_integ_1cm = data_offbeam_light_evt_chi2_proton_cut.sum_integ_1cm.fillna(0.)
data_offbeam_light_evt_chi2_proton_cut.sum_integ_2cm = data_offbeam_light_evt_chi2_proton_cut.sum_integ_2cm.fillna(0.)
data_offbeam_light_evt_chi2_proton_cut.sum_integ_3cm = data_offbeam_light_evt_chi2_proton_cut.sum_integ_3cm.fillna(0.)
data_offbeam_light_evt_chi2_proton_cut.sum_integ_4cm = data_offbeam_light_evt_chi2_proton_cut.sum_integ_4cm.fillna(0.)

data_bnb_light_evt_chi2_proton_cut.sum_integ_1cm = data_bnb_light_evt_chi2_proton_cut.sum_integ_1cm.fillna(0.)
data_bnb_light_evt_chi2_proton_cut.sum_integ_2cm = data_bnb_light_evt_chi2_proton_cut.sum_integ_2cm.fillna(0.)
data_bnb_light_evt_chi2_proton_cut.sum_integ_3cm = data_bnb_light_evt_chi2_proton_cut.sum_integ_3cm.fillna(0.)
data_bnb_light_evt_chi2_proton_cut.sum_integ_4cm = data_bnb_light_evt_chi2_proton_cut.sum_integ_4cm.fillna(0.)

In [ ]:
mc_evt_chi2_proton_cut.sum_integ_2cm

In [ ]:
mc_evt_chi2_proton_cut

In [ ]:
sum_integ_1cm_col = ('sum_integ_1cm', '')
sum_integ_2cm_col = ('sum_integ_2cm', '')
sum_integ_3cm_col = ('sum_integ_3cm', '')
sum_integ_4cm_col = ('sum_integ_4cm', '')

draw_mc_data_shape_comp_per_slc_blind(mc_evt_chi2_proton_cut, data_offbeam_light_evt_chi2_proton_cut, data_bnb_light_evt_chi2_proton_cut, sum_integ_1cm_col, "Sum Integ. 1cm",  "A.U.", 0, 30000, 45, "./no_dirz_cut/j_chi2_proton_sum_integ_1cm", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_chi2_proton_cut, data_offbeam_light_evt_chi2_proton_cut, data_bnb_light_evt_chi2_proton_cut, sum_integ_2cm_col, "Sum Integ. 2cm",  "A.U.", 0, 50000, 50, "./no_dirz_cut/j_chi2_proton_sum_integ_2cm", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_chi2_proton_cut, data_offbeam_light_evt_chi2_proton_cut, data_bnb_light_evt_chi2_proton_cut, sum_integ_3cm_col, "Sum Integ. 3cm",  "A.U.", 0, 50000, 50, "./no_dirz_cut/j_chi2_proton_sum_integ_3cm", False, True, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_chi2_proton_cut, data_offbeam_light_evt_chi2_proton_cut, data_bnb_light_evt_chi2_proton_cut, sum_integ_4cm_col, "Sum Integ. 4cm",  "A.U.", 0, 50000, 50, "./no_dirz_cut/j_chi2_proton_sum_integ_4cm", False, True, blind=True, blind_req=0.1)

In [ ]:
## 1cm : 4000, 2 cm : 7000, 3 cm : 10000, 4cm : 13000
mc_evt_charge_sphere_cut = mc_evt_chi2_proton_cut[(mc_evt_chi2_proton_cut.sum_integ_1cm < 4000) & (mc_evt_chi2_proton_cut.sum_integ_2cm < 7000) & (mc_evt_chi2_proton_cut.sum_integ_3cm < 10000) & (mc_evt_chi2_proton_cut.sum_integ_4cm < 13000)]
data_offbeam_light_evt_charge_sphere_cut = data_offbeam_light_evt_chi2_proton_cut[(data_offbeam_light_evt_chi2_proton_cut.sum_integ_1cm < 4000) & (data_offbeam_light_evt_chi2_proton_cut.sum_integ_2cm < 7000) & (data_offbeam_light_evt_chi2_proton_cut.sum_integ_3cm < 10000) & (data_offbeam_light_evt_chi2_proton_cut.sum_integ_4cm < 13000)]
data_bnb_light_evt_charge_sphere_cut = data_bnb_light_evt_chi2_proton_cut[(data_bnb_light_evt_chi2_proton_cut.sum_integ_1cm < 4000) & (data_bnb_light_evt_chi2_proton_cut.sum_integ_2cm < 7000) & (data_bnb_light_evt_chi2_proton_cut.sum_integ_3cm < 10000) & (data_bnb_light_evt_chi2_proton_cut.sum_integ_4cm < 13000)]

In [ ]:
mc_charge_spere_3_over_4m3 = mc_evt_charge_sphere_cut.sum_integ_3cm / (mc_evt_charge_sphere_cut.sum_integ_4cm - mc_evt_charge_sphere_cut.sum_integ_3cm)
data_offbeam_charge_sphere_3_over_4m3 = data_offbeam_light_evt_charge_sphere_cut.sum_integ_3cm / (data_offbeam_light_evt_charge_sphere_cut.sum_integ_4cm - data_offbeam_light_evt_charge_sphere_cut.sum_integ_3cm)
data_bnb_charge_sphere_3_over_4m3 = data_bnb_light_evt_charge_sphere_cut.sum_integ_3cm / (data_bnb_light_evt_charge_sphere_cut.sum_integ_4cm - data_bnb_light_evt_charge_sphere_cut.sum_integ_3cm)

mc_charge_spere_3_over_4m3 = mc_charge_spere_3_over_4m3.fillna(0.)
data_offbeam_charge_sphere_3_over_4m3 = data_offbeam_charge_sphere_3_over_4m3.fillna(0.)
data_bnb_charge_sphere_3_over_4m3 = data_bnb_charge_sphere_3_over_4m3.fillna(0.)

In [ ]:
mc_evt_charge_sphere_cut[('charge_spere_3_over_4m3', '')] = mc_charge_spere_3_over_4m3
data_offbeam_light_evt_charge_sphere_cut[('charge_spere_3_over_4m3', '')] = data_offbeam_charge_sphere_3_over_4m3
data_bnb_light_evt_charge_sphere_cut[('charge_spere_3_over_4m3', '')] = data_bnb_charge_sphere_3_over_4m3

In [ ]:
draw_mc_data_shape_comp_per_slc_blind(mc_evt_charge_sphere_cut, data_offbeam_light_evt_charge_sphere_cut, data_bnb_light_evt_charge_sphere_cut, ('charge_spere_3_over_4m3', ''), "Charge Ratio (3cm / (4cm-3cm))",  "A.U.", 0, 40, 50, "./no_dirz_cut/k_charge_sphere_cut_charge_3_over_4m3", False, True, blind=True, blind_req=0.1)

In [ ]:
mc_evt_charge_sphere_ratio_cut = mc_evt_charge_sphere_cut[mc_evt_charge_sphere_cut.charge_spere_3_over_4m3 < 6.0]
data_offbeam_light_evt_charge_sphere_ratio_cut = data_offbeam_light_evt_charge_sphere_cut[data_offbeam_light_evt_charge_sphere_cut.charge_spere_3_over_4m3 < 6.0]
data_bnb_light_evt_charge_sphere_ratio_cut = data_bnb_light_evt_charge_sphere_cut[data_bnb_light_evt_charge_sphere_cut.charge_spere_3_over_4m3 < 6.0]

In [ ]:
reco_t_col = ('reco_t', '')

In [ ]:
draw_mc_data_shape_comp_per_slc_blind(mc_evt_charge_sphere_ratio_cut, data_offbeam_light_evt_charge_sphere_ratio_cut, data_bnb_light_evt_charge_sphere_ratio_cut, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0, 0.5, 25, "./no_dirz_cut/l_charge_sphere_ratio_reco_t_full", False, False, blind=True, blind_req=0.1) 
draw_mc_data_shape_comp_per_slc_blind(mc_evt_charge_sphere_ratio_cut, data_offbeam_light_evt_charge_sphere_ratio_cut, data_bnb_light_evt_charge_sphere_ratio_cut, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0, 0.04, 4, "./no_dirz_cut/l_charge_sphere_ratio_reco_t_signal_region", False, False, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_charge_sphere_ratio_cut, data_offbeam_light_evt_charge_sphere_ratio_cut, data_bnb_light_evt_charge_sphere_ratio_cut, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0.06, 0.5, 10, "./no_dirz_cut/l_charge_sphere_ratio_reco_t_sideband", False, False, blind=True, blind_req=0.1) 

In [ ]:
mc_evt_charge_sphere_ratio_cut

In [ ]:
mc_evt_charge_sphere_ratio_cut_signal = mc_evt_charge_sphere_ratio_cut[mc_evt_charge_sphere_ratio_cut.nuint_categ == 1]

### Containment

In [ ]:
mc_evt_contained_cut_muon_only = mc_evt_charge_sphere_ratio_cut[(mc_evt_charge_sphere_ratio_cut.is_muon_contained == 1)]
data_offbeam_contained_cut_muon_only = data_offbeam_light_evt_charge_sphere_ratio_cut[(data_offbeam_light_evt_charge_sphere_ratio_cut.is_muon_contained == 1)]
data_bnb_contained_cut_muon_only = data_bnb_light_evt_charge_sphere_ratio_cut[(data_bnb_light_evt_charge_sphere_ratio_cut.is_muon_contained == 1)]

In [ ]:
draw_mc_data_shape_comp_per_slc_blind(mc_evt_contained_cut_muon_only, data_offbeam_contained_cut_muon_only, data_bnb_contained_cut_muon_only, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0, 0.5, 10, "./no_dirz_cut/l_a_muon_containment_reco_t_full", False, False, blind=True, blind_req=0.1) 
draw_mc_data_shape_comp_per_slc_blind(mc_evt_contained_cut_muon_only, data_offbeam_contained_cut_muon_only, data_bnb_contained_cut_muon_only, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0, 0.04, 4, "./no_dirz_cut/l_a_muon_containment_reco_t_signal_region", False, False, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_contained_cut_muon_only, data_offbeam_contained_cut_muon_only, data_bnb_contained_cut_muon_only, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0.06, 0.5, 5, "./no_dirz_cut/l_a_muon_containment_reco_t_sideband", False, False, blind=True, blind_req=0.1)

In [ ]:
mc_evt_contained_cut = mc_evt_charge_sphere_ratio_cut[(mc_evt_charge_sphere_ratio_cut.is_muon_contained) & (mc_evt_charge_sphere_ratio_cut.is_cpi_contained)]
data_offbeam_light_evt_contained_cut = data_offbeam_light_evt_charge_sphere_ratio_cut[(data_offbeam_light_evt_charge_sphere_ratio_cut.is_muon_contained) & (data_offbeam_light_evt_charge_sphere_ratio_cut.is_cpi_contained)]
data_bnb_light_evt_contained_cut = data_bnb_light_evt_charge_sphere_ratio_cut[(data_bnb_light_evt_charge_sphere_ratio_cut.is_muon_contained) & (data_bnb_light_evt_charge_sphere_ratio_cut.is_cpi_contained)]

In [ ]:
draw_mc_data_shape_comp_per_slc_blind(mc_evt_contained_cut, data_offbeam_light_evt_contained_cut, data_bnb_light_evt_contained_cut, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0, 0.5, 10, "./no_dirz_cut/m_containment_reco_t_full", False, False, blind=True, blind_req=0.1) 
draw_mc_data_shape_comp_per_slc_blind(mc_evt_contained_cut, data_offbeam_light_evt_contained_cut, data_bnb_light_evt_contained_cut, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0, 0.04, 4, "./no_dirz_cut/m_containment_reco_t_signal_region", False, False, blind=True, blind_req=0.1)
draw_mc_data_shape_comp_per_slc_blind(mc_evt_contained_cut, data_offbeam_light_evt_contained_cut, data_bnb_light_evt_contained_cut, reco_t_col, r"$|t|_{reco.}$ [$(GeV/c)^2$]",  "A.U.", 0.06, 0.5, 5, "./no_dirz_cut/m_containment_reco_t_sideband", False, False, blind=True, blind_req=0.1) 

### SR and SB x-sec var plots

In [ ]:
mc_sr = mc_evt_charge_sphere_ratio_cut[mc_evt_charge_sphere_ratio_cut.reco_t < 0.04]
mc_sb = mc_evt_charge_sphere_ratio_cut[mc_evt_charge_sphere_ratio_cut.reco_t > 0.06]

data_bnb_light_sr = data_bnb_light_evt_charge_sphere_ratio_cut[data_bnb_light_evt_charge_sphere_ratio_cut.reco_t < 0.04]
data_bnb_light_sb = data_bnb_light_evt_charge_sphere_ratio_cut[data_bnb_light_evt_charge_sphere_ratio_cut.reco_t > 0.06]

data_offbeam_light_sr = data_offbeam_light_evt_charge_sphere_ratio_cut[data_offbeam_light_evt_charge_sphere_ratio_cut.reco_t < 0.04]
data_offbeam_light_sb = data_offbeam_light_evt_charge_sphere_ratio_cut[data_offbeam_light_evt_charge_sphere_ratio_cut.reco_t > 0.06]

mc_sr_contained = mc_sr[(mc_sr.is_muon_contained) & (mc_sr.is_cpi_contained)]
mc_sb_contained = mc_sb[(mc_sb.is_muon_contained) & (mc_sb.is_cpi_contained)]

data_bnb_light_sr_contained = data_bnb_light_sr[(data_bnb_light_sr.is_muon_contained) & (data_bnb_light_sr.is_cpi_contained)]
data_bnb_light_sb_contained = data_bnb_light_sb[(data_bnb_light_sb.is_muon_contained) & (data_bnb_light_sb.is_cpi_contained)]

data_offbeam_light_sr_contained = data_offbeam_light_sr[(data_offbeam_light_sr.is_muon_contained) & (data_offbeam_light_sr.is_cpi_contained)]
data_offbeam_light_sb_contained = data_offbeam_light_sb[(data_offbeam_light_sb.is_muon_contained) & (data_offbeam_light_sb.is_cpi_contained)]

In [ ]:
range_p_mu_col = ('range_p_mu', '')
draw_mc_data_shape_comp_per_slc_blind(mc_sr, data_offbeam_light_sr, data_bnb_light_sr, range_p_mu_col, r"$P_{\mu^{\mp}}^{reco}~ [GeV/c]$",  "A.U.", 0, 1.5, 5, "./no_dirz_cut/l_charge_sphere_ratio_p_mu_signal_region", False, False, blind=True, blind_req=0.1, label_top_right="Signal Region")
draw_mc_data_shape_comp_per_slc_blind(mc_sb, data_offbeam_light_sb, data_bnb_light_sb, range_p_mu_col, r"$P_{\mu^{\mp}}^{reco}~ [GeV/c]$",  "A.U.", 0, 1.5, 5, "./no_dirz_cut/l_charge_sphere_ratio_p_mu_sideband", False, False, blind=True, blind_req=0.1, label_top_right="Sideband")
draw_mc_data_shape_comp_per_slc_blind(mc_sr_contained, data_offbeam_light_sr_contained, data_bnb_light_sr_contained, range_p_mu_col, r"$P_{\mu^{\mp}}^{reco}~ [GeV/c]$",  "A.U.", 0, 1.5, 5, "./no_dirz_cut/l_charge_sphere_ratio_p_mu_signal_region_contained", False, False, blind=True, blind_req=0.1, label_top_right="Signal Region Contained")
draw_mc_data_shape_comp_per_slc_blind(mc_sb_contained, data_offbeam_light_sb_contained, data_bnb_light_sb_contained, range_p_mu_col, r"$P_{\mu^{\mp}}^{reco}~ [GeV/c]$",  "A.U.", 0, 1.5, 5, "./no_dirz_cut/l_charge_sphere_ratio_p_mu_sideband_contained", False, False, blind=True, blind_req=0.1, label_top_right="Sideband Contained")


In [ ]:

draw_mc_data_shape_comp_per_slc_blind(mc_sr, data_offbeam_light_sr, data_bnb_light_sr, long_dirz_col, r"$cos \theta_{\mu^{\mp}}^{reco}$",  "A.U.", -1, 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_mu_signal_region", False, True, blind=True, blind_req=0.1, label_top_right="Signal Region")
draw_mc_data_shape_comp_per_slc_blind(mc_sb, data_offbeam_light_sb, data_bnb_light_sb, long_dirz_col, r"$cos \theta_{\mu^{\mp}}^{reco}$",  "A.U.", -1, 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_mu_sideband", False, False, blind=True, blind_req=0.1, label_top_right="Sideband")
draw_mc_data_shape_comp_per_slc_blind(mc_sr_contained, data_offbeam_light_sr_contained, data_bnb_light_sr_contained, long_dirz_col, r"$cos \theta_{\mu^{\mp}}^{reco}$",  "A.U.", -1., 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_mu_signal_region_contained", False, True, blind=True, blind_req=0.1, label_top_right="Signal Region Contained")
draw_mc_data_shape_comp_per_slc_blind(mc_sb_contained, data_offbeam_light_sb_contained, data_bnb_light_sb_contained, long_dirz_col, r"$cos \theta_{\mu^{\mp}}^{reco}$",  "A.U.", -1, 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_mu_sideband_contained", False, False, blind=True, blind_req=0.1, label_top_right="Sideband Contained")


In [ ]:

draw_mc_data_shape_comp_per_slc_blind(mc_sr, data_offbeam_light_sr, data_bnb_light_sr, short_dirz_col, r"$cos \theta_{\pi^\pm}^{reco}$",  "A.U.", -1., 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_pi_signal_region", False, True, blind=True, blind_req=0.1, label_top_right="Signal Region")
draw_mc_data_shape_comp_per_slc_blind(mc_sb, data_offbeam_light_sb, data_bnb_light_sb, short_dirz_col, r"$cos \theta_{\pi^\pm}^{reco}$",  "A.U.", -1., 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_pi_sideband", False, False, blind=True, blind_req=0.1, label_top_right="Sideband")
draw_mc_data_shape_comp_per_slc_blind(mc_sr_contained, data_offbeam_light_sr_contained, data_bnb_light_sr_contained, short_dirz_col, r"$cos \theta_{\pi^\pm}^{reco}$",  "A.U.", -1., 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_pi_signal_region_contained", False, True, blind=True, blind_req=0.1, label_top_right="Signal Region Contained")
draw_mc_data_shape_comp_per_slc_blind(mc_sb_contained, data_offbeam_light_sb_contained, data_bnb_light_sb_contained, short_dirz_col, r"$cos \theta_{\pi^\pm}^{reco}$",  "A.U.", -1., 1., 4, "./no_dirz_cut/l_charge_sphere_ratio_cos_pi_sideband_contained", False, False, blind=True, blind_req=0.1, label_top_right="Sideband Contained")


### efficiency and purity plots

In [ ]:
cutflow_dfs = [mc_bnb_cosmic_dfs["evt"], mc_evt_pe_2000, mc_bcfm_score_0p05, mc_evt_n_prong_cut, mc_evt_n_trk_cut, mc_evt_n_proton_cut, mc_evt_n_shower_cut,
               mc_evt_dirz_cut, mc_evt_dirz_cut[mc_evt_dirz_cut.reco_t < 0.04],
               mc_evt_open_ang_cut, mc_evt_open_ang_cut[mc_evt_open_ang_cut.reco_t < 0.04],
               mc_evt_chi2_proton_cut, mc_evt_chi2_proton_cut[mc_evt_chi2_proton_cut.reco_t < 0.04],
               mc_evt_charge_sphere_cut, mc_evt_charge_sphere_cut[mc_evt_charge_sphere_cut.reco_t < 0.04],
               mc_evt_charge_sphere_ratio_cut, mc_evt_charge_sphere_ratio_cut[mc_evt_charge_sphere_ratio_cut.reco_t < 0.04],
               mc_evt_contained_cut_muon_only, mc_evt_contained_cut_muon_only[mc_evt_contained_cut_muon_only.reco_t < 0.04],
               mc_evt_contained_cut, mc_evt_contained_cut[mc_evt_contained_cut.reco_t < 0.04]]
cutflow_lables = ["Not clear cosmic \& FV", "Total PE $>$ 1500", "BCFM Score $>$ 0.05", "N(Prongs) == 2", "N(Tracks) == 2", "N(Protons) == 0", "N(Showers) == 0",
                  r"$cos \theta_{\mu^\mp} > 0.7$ \& $cos \theta_{\pi^\pm} > 0.5$", "-- In signal region",
                  r"$cos \theta_{\mu^\mp \pi^\pm} > 0.5$", "-- In signal region",
                  r"$\chi^2_p(\mu^\mp) > 100$ \& $\chi^2_p(\pi^\pm) > 75$", "-- In signal region",
                  "Charge Sphere Cuts", "-- In signal region",
                  "Charge Sphere Ratio Cut", "-- In signal region",
                  "Muon is Contained", "-- In signal region",
                  "Muon & Pion Contained", "-- In signal region",]
signal_true_nudf = mc_bnb_cosmic_dfs["mcnu"][mc_bnb_cosmic_dfs["mcnu"].nuint_categ == 1]

In [ ]:
signal_true_nudf

In [ ]:
def make_purity_eff_latex_table(cutflow_dfs, cutflow_lables, signal_true_nudf):
    """
    Creates a latex cutflow table showing the number of events, purity, and efficiency at each cut stage and saves it as a .tex file.
    """
    # Initialize lists to store cutflow information
    cut_names = []
    event_counts = []
    purities = []
    efficiencies = []
    
    # Get the total number of signal events for efficiency calculation
    total_signal_events = len(signal_true_nudf)
    
    for df, label in zip(cutflow_dfs, cutflow_lables):
        cut_names.append(label)
        event_count = len(df)
        event_counts.append(event_count)
        
        # Calculate purity: (Number of signal events) / (Total events after cut)
        signal_events_after_cut = len(df[df.nuint_categ == 1])
        purity = signal_events_after_cut / event_count if event_count > 0 else 0
        purities.append(purity)
        
        # Calculate efficiency: (Number of signal events after cut) / (Total signal events before cuts)
        signal_events_after_cut = df[df.nuint_categ == 1].sort_values('tmatch_eff', ascending=False).groupby(level=['__ntuple', 'entry']).head(1)
        efficiency = len(signal_events_after_cut) / total_signal_events if total_signal_events > 0 else 0
        efficiencies.append(efficiency)
    
    # Create a DataFrame for the cutflow table
    cutflow_table_df = pd.DataFrame({
        'Cut': cut_names,
        'Events': event_counts,
        'Purity': purities,
        'Efficiency': efficiencies
    })
    
    # Convert the DataFrame to a LaTeX table
    latex_table = cutflow_table_df.to_latex(index=False, float_format="%.4f")
    
    # Save the LaTeX table to a .tex file
    with open("cutflow_table.tex", "w") as f:
        f.write(latex_table)


In [ ]:
make_purity_eff_latex_table(cutflow_dfs, cutflow_lables, signal_true_nudf)